# 03c - RM-c: frozen encoder + Retrieval-Augmented Classification

RM-c tidak melatih apa pun. Ia memakai ulang dua artefak milik RM-b, yaitu
embedding beku dan head yang sudah terlatih, lalu menambahkan cabang retrieval.

Alur per sampel:

1. `p_bert` = softmax(head(embedding))
2. Cari k tetangga terdekat di indeks FAISS yang dibangun HANYA dari split train,
   lalu ubah label tetangga menjadi distribusi `p_retr`
3. `p_final = (1 - alpha) * p_bert + alpha * p_retr`, prediksi = argmax

Fusi dilakukan pada level probabilitas dan softmax hanya diterapkan sekali, di
cabang BERT sebelum fusi. Karena kedua masukan sudah berupa distribusi dan bobot
fusinya berjumlah satu, hasilnya sudah menjadi distribusi sah; softmax kedua
hanya akan meratakan selisih dan bisa mengubah argmax pada kasus nyaris seri.

Notebook ini menjalankan RM-c dalam DUA LAPIS.

- **RM-c standar**: fusi linear di atas head juara RM-b saja (`alpha x k`,
  `tuning_grids/RMC_TUNING_GRID.csv`, 66 konfigurasi).
- **Eksplorasi RM-c**: fusi linear yang SAMA di atas SEMUA head RM-b (setiap
  head yang disimpan `03b_rmb_frozen.ipynb`), ruang `alpha x k` yang sama
  (`tuning_grids/RMC_EXPLORATION_GRID.csv`, 61 konfigurasi per head: 66 dikurangi
  lima baris alpha=0 yang identik untuk semua k).

Kedua lapis memakai satu rumus saja, yaitu fusi linear di atas.

Tidak ada training di kedua lapis: head dimuat dari checkpoint, indeks FAISS
dibangun sekali dari train, dan split test tidak dibuka. Juara eksplorasi
menggantikan juara standar hanya bila selisihnya melampaui ambang seri DAN
lolos bootstrap berpasangan pada validation.

Prasyarat: `03b_rmb_frozen.ipynb` sudah dijalankan (butuh seluruh state head di
`checkpoints/rmb_heads/` dan `rmb_best.pt`).


In [ ]:
from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.default_out_dir

runner = CampaignRunner(out_dir=OUT_DIR)
print("device        :", runner.device)
print("encoder       :", runner.model_name)
print("keluaran      :", runner.out_dir)
print("train/val/test:", [len(frame) for frame in runner.data.frames.values()])


## 0. Pemulihan checkpoint

Checkpoint tidak ikut git (`outputs/**/checkpoints/` di-gitignore), sedangkan
`best.json` dan riwayat run ikut. Di clone atau instance baru `rmb_best.pt`
dan `rmb_heads/` tidak ada, dan menjalankan ulang 03b tidak membuatnya kembali:
F1 yang sama dengan juara tidak dipromosikan sehingga tidak disimpan. Sel di
bawah membangunnya dari konfigurasi yang tercatat di `runs_rmb.csv` tanpa
mengubah riwayat atau `best.json`. Bila checkpoint sudah ada, sel ini tidak
melakukan apa-apa.


In [ ]:
dipulihkan = runner.restore_checkpoints()
print(dipulihkan)


## 1. Head RM-b yang diwarisi

In [ ]:
head, head_config = runner._load_best_head()
print("konfigurasi head RM-b terbaik:", head_config)
print("indeks FAISS akan dibangun dari", len(runner.features.labels["train"]), "vektor train")


Indeks dibangun eksklusif dari split train. Kalau val atau test ikut masuk,
retrieval akan menemukan sampel uji di dalam indeksnya sendiri dan hasilnya
kehilangan makna.


## 2. Muat rancangan grid standar

In [ ]:
import pandas as pd

GRID_DIR = settings.data_dir.parent / "tuning_grids"
RMC_GRIDS = ("RMC_TUNING_GRID.csv", "RMC_TUNING_GRID_STAGE2.csv")


def muat_grid(nama: str) -> list[dict]:
    frame = pd.read_csv(GRID_DIR / nama)
    catatan = frame.pop("catatan") if "catatan" in frame.columns else ""
    return [
        {"config": {k: v for k, v in baris.items() if pd.notna(v)},
         "note": catatan.iloc[i] if hasattr(catatan, "iloc") else ""}
        for i, baris in enumerate(frame.to_dict("records"))
    ]


for berkas in RMC_GRIDS:
    print(f"  {berkas}: {len(pd.read_csv(GRID_DIR / berkas))} konfigurasi")


## 3. RM-c standar: fusi linear di atas head juara RM-b

RM-c mewarisi head RM-b terbaik, jadi kampanye ini harus dijalankan SETELAH
RM-b selesai. Karena tidak ada training sama sekali, seluruh kombinasi
`alpha x k` selesai dalam hitungan detik.


In [ ]:
batch_rmc = []
for berkas in RMC_GRIDS:
    batch_id = berkas.replace(".csv", "").lower()
    runner.run_batch("rmc", muat_grid(berkas), batch_id=batch_id)
    batch_rmc.append(batch_id)

riwayat_rmc = runner.reporter.runs_frame("rmc")
riwayat_rmc[riwayat_rmc["batch_id"].isin(batch_rmc)].nlargest(10, "val_f1_macro")[
    ["run_id", "alpha", "k", "weighting", "val_f1_macro", "val_f1_judi", "eval_time_s"]
]


Baris dengan `alpha=0` (bila ada di grid) harus identik dengan F1 RM-b murni,
dan `alpha=1` membuang cabang BERT sepenuhnya. Kedua ujung itu berfungsi
sebagai pemeriksaan kewarasan untuk mekanisme fusi.


## 4. Eksplorasi RM-c: seluruh head RM-b x alpha x k

RM-c standar menguji satu head. Bagian ini menguji fusi linear yang sama di atas
SETIAP head RM-b dengan `alpha x k` yang disapu, lihat
`tuning_grids/RMC_EXPLORATION_GRID.md`. Tidak ada pelatihan: head dimuat dari
`checkpoints/rmb_heads/`, indeks FAISS hanya dari train, dan split test tidak
dibuka.

Aturan seleksi: F1-macro validation, F1 judi sebagai pemecah seri, selisih di
bawah 0,15 pp dihitung seri sehingga head yang lebih murah menang. Karena
kandidatnya ratusan, pemenangnya rawan bias seleksi; penantang baru hanya
menggantikan juara standar bila selisihnya melampaui 0,15 pp DAN lolos
bootstrap berpasangan pada split validation.


In [ ]:
from src.services.rmc_exploration import load_exploration_grid

grid_eksplorasi = load_exploration_grid(GRID_DIR / "RMC_EXPLORATION_GRID.csv")
eksplorasi = runner.explore_rmc(grid_eksplorasi)

print(f"{len(grid_eksplorasi)} konfigurasi alpha x k per head, "
      f"{len(eksplorasi['runs'])} evaluasi seluruhnya")
eksplorasi["per_formula"].round(4)


`shared_*` adalah konfigurasi SERAGAM terbaik, yaitu (alpha, k) dengan rata-rata
F1-macro tertinggi lintas semua head. Angka ini menjawab apakah RAC membantu tanpa
disetel per head, dan lebih jujur daripada `best_*` yang dipilih dari puluhan
kandidat per head.


In [ ]:
per_head = eksplorasi["per_head"]
print(f"RAC membantu di {per_head['rac_helps'].sum()} dari {len(per_head)} head "
      f"(kenaikan di atas ambang seri {settings.tie_threshold_pp} pp)")

per_head[[
    "rmb_run_id", "head_arch", "hidden_dim", "epochs", "lr", "head_params",
    "val_f1_rmb", "best_alpha", "best_k", "val_f1_rac_best",
    "gain_best_pp", "on_pareto",
]].round(4)


In [ ]:
eksplorasi["runs"].nlargest(10, ["val_f1_macro", "val_f1_judi"])[
    ["rmb_run_id", "alpha", "k", "weighting", "val_f1_macro",
     "val_f1_judi", "gain_pp"]
].round(4)


In [ ]:
import matplotlib.pyplot as plt

fig, (kiri, kanan) = plt.subplots(1, 2, figsize=(12, 4.5))

kiri.scatter(per_head["val_f1_rmb"], per_head["gain_best_pp"])
kiri.axhline(settings.tie_threshold_pp, color="gray", linestyle="--", label="ambang seri")
kiri.axhline(0, color="black", linewidth=0.8)
kiri.set_xlabel("F1-macro head RM-b sendiri (validation)")
kiri.set_ylabel("kenaikan terbaik dari RAC (pp)")
kiri.set_title("Kenaikan RAC vs kekuatan head")
kiri.legend(fontsize=8)
kiri.grid(alpha=0.3)

kurva = eksplorasi["runs"].groupby(["rmb_run_id", "alpha"])["gain_pp"].max().unstack("alpha")
for _, baris in kurva.iterrows():
    kanan.plot(kurva.columns, baris.values, color="tab:blue", alpha=0.25)
kanan.plot(kurva.columns, kurva.mean(), color="tab:red", linewidth=2.5, label="rata-rata head")
kanan.axhline(0, color="black", linewidth=0.8)
kanan.set_xlabel("alpha (fusi linear)")
kanan.set_ylabel("kenaikan dari RAC (pp), k terbaik per alpha")
kanan.set_title("Kurva alpha per head")
kanan.legend(fontsize=8)
kanan.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(runner.exploration_dir / "rmc_exploration.png", dpi=150)
plt.show()


### Putusan juara RM-c

Penantang adalah konfigurasi terbaik eksplorasi menurut aturan seleksi di atas.
Ia dibandingkan dengan juara standar pada prediksi validation yang sama lewat
bootstrap berpasangan. Bila menang, `best.json` dan `checkpoints/rmc_best.pt`
diganti; `rmc_best.pt` lalu memuat head dan konfigurasi fusi penantang, dan
`05_final_benchmark.ipynb` memuat RM-c dari sana. Bila kalah, juara standar
tetap. Keduanya hasil sah; catatannya di `rmc_exploration/champion_decision.json`.


In [ ]:
keputusan = runner.decide_rmc_champion(eksplorasi["runs"])

print(f"pemenang     : {keputusan['winner']}")
print(f"penantang    : {keputusan['challenger']}")
print(f"petahana     : {keputusan['incumbent']}")
print(f"selisih F1   : {keputusan['delta_pp']:+.3f} pp "
      f"(CI95 {keputusan['ci95_pp'][0]:+.3f} sampai {keputusan['ci95_pp'][1]:+.3f}; "
      f"ambang seri {keputusan['tie_threshold_pp']} pp)")


## Ringkasan

RM-c tidak melatih apa pun, tetapi memakai head RM-b, sehingga biaya latihnya
adalah biaya head itu (tercatat di `best.json` sebagai `head_train_time_s` dan
`head_trainable_params`). Biaya tambahannya ada di inferensi: pembangunan
indeks sekali dan penelusuran k tetangga per prediksi, diukur terpisah di
`05_final_benchmark.ipynb`.

Ketiga skenario sudah punya juara di `best.json`. Split test masih tertutup
dan baru dibuka satu kali di `05_final_benchmark.ipynb`.
